In [1]:
import os
import pandas as pd

import requests
from io import StringIO
from io import BytesIO

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load env variables
load_dotenv()

host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
database = os.getenv("NEW_DATABASE")

In [2]:
# -----------------------------
# CREATE DATABASE
# -----------------------------

default_db_url = (
    f"postgresql+psycopg://{user}:{password}@{host}:{port}/postgres"
)

default_engine = create_engine(default_db_url)

with default_engine.connect() as conn:

    conn.execution_options(isolation_level="AUTOCOMMIT")

    result = conn.execute(
        text(f"SELECT 1 FROM pg_database WHERE datname='{database}'")
    )

    if not result.scalar():

        conn.execute(text(f'CREATE DATABASE "{database}"'))

        print(f"Database '{database}' created.")


In [4]:
# -----------------------------
# CONNECT TO NEW DATABASE
# -----------------------------

db_url = (
    f"postgresql+psycopg://{user}:{password}@{host}:{port}/{database}"
)

engine = create_engine(db_url)

In [11]:
# SQL query
drop_table_sql = f"""
DROP TABLE IF EXISTS orders, people, products, returned_orders;
"""

# Execute query
with engine.connect() as conn:

    conn.execute(text(drop_table_sql))

    conn.commit()

print(f"Tables deleted successfully.")

Tables deleted successfully.


In [8]:
# Download the dataset

#orders_url="https://project-dataset7901.s3.us-east-1.amazonaws.com/postgresql-sales-data/orders.csv"
#orders_df = pd.read_csv(orders_url)
#orders_df.to_csv('dataset/orders.csv')

#people_url="https://project-dataset7901.s3.us-east-1.amazonaws.com/postgresql-sales-data/people.csv"
#people_df = pd.read_csv(people_url)
#people_df.to_csv('dataset/people.csv')

#products_url="https://project-dataset7901.s3.us-east-1.amazonaws.com/postgresql-sales-data/products.csv"
#products_df = pd.read_csv(products_url)
#products_df.to_csv('dataset/products.csv')

#returned_orders_url="https://project-dataset7901.s3.us-east-1.amazonaws.com/postgresql-sales-data/returned_orders.csv"
#returned_orders_df = pd.read_csv(returned_orders_url)
#returned_orders_df.to_csv('dataset/returned_orders.csv')

In [12]:

# -----------------------------
# READ AND CLEAN CSV DATA
# -----------------------------
orders_df = pd.read_csv('dataset/orders.csv')

# Clean column names, just in case
orders_df.columns = (
    orders_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# Convert datetime columns
orders_df["ship_date"] = pd.to_datetime(orders_df["ship_date"])
orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])

orders_df.head()


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,city,state,...,postal_code,market,region,product_id,sales,quantity,discount,profit,shipping_cost,order_priority
0,957,MX-2014-105921,2014-05-28,2014-06-03,Standard Class,ZC-21910,Zuschuss Carroll,Consumer,San Salvador,San Salvador,...,NaN,LATAM,Central,TEC-AC-10004626,342.080,2.0,0.00,0.0000,21.713,Medium
1,24359,ID-2013-61442,2013-01-15,2013-01-21,Standard Class,JB-16000,Joy Bell-,Consumer,Manila,National Capital,...,NaN,APAC,Southeast Asia,OFF-BI-10001400,122.400,5.0,0.15,0.0000,21.710,Low
2,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,10024.0,US,East,TEC-AC-10003033,2309.650,7.0,0.00,762.1845,933.570,Critical
3,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,NaN,APAC,Oceania,FUR-CH-10003950,3709.395,9.0,0.10,-288.7650,923.630,Critical
4,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,NaN,APAC,Oceania,TEC-PH-10004664,5175.171,9.0,0.10,919.9710,915.490,Medium


In [13]:
# -----------------------------
# CREATE TABLE
# -----------------------------

# Create table
create_table_sql = """
CREATE TABLE orders (
    row_id INTEGER PRIMARY KEY,
    order_id VARCHAR(20),
    order_date DATE,
    ship_date DATE,
    ship_mode VARCHAR(50),
    customer_id VARCHAR(20),
    customer_name VARCHAR(100),
    segment VARCHAR(50),
    city VARCHAR(100),
    state VARCHAR(100),
    country VARCHAR(100),
    postal_code NUMERIC,
    market VARCHAR(50),
    region VARCHAR(100),
    product_id VARCHAR(30),
    sales NUMERIC(12,2),
    quantity INTEGER,
    discount NUMERIC(5,2),
    profit NUMERIC(12,4),
    shipping_cost NUMERIC(12,3),
    order_priority VARCHAR(20)
);
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()

# Insert CSV data
orders_df.to_sql(
    "orders",
    engine,
    if_exists="append",
    index=False
)

print("CSV data loaded into orders table successfully.")

CSV data loaded into orders table successfully.


In [14]:
query = "SELECT * FROM orders LIMIT 5;"

df_check = pd.read_sql(query, engine)

print(df_check)

   row_id        order_id  order_date   ship_date       ship_mode customer_id  \
0     957  MX-2014-105921  2014-05-28  2014-06-03  Standard Class    ZC-21910   
1   24359   ID-2013-61442  2013-01-15  2013-01-21  Standard Class    JB-16000   
2   32298  CA-2012-124891  2012-07-31  2012-07-31        Same Day    RH-19495   
3   26341   IN-2013-77878  2013-02-05  2013-02-07    Second Class    JR-16210   
4   25330   IN-2013-71249  2013-10-17  2013-10-18     First Class    CR-12730   

      customer_name    segment           city             state  ...  \
0  Zuschuss Carroll   Consumer   San Salvador      San Salvador  ...   
1         Joy Bell-   Consumer         Manila  National Capital  ...   
2       Rick Hansen   Consumer  New York City          New York  ...   
3     Justin Ritter  Corporate     Wollongong   New South Wales  ...   
4      Craig Reiter   Consumer       Brisbane        Queensland  ...   

  postal_code  market          region       product_id    sales  quantity  \
0  

In [15]:
# -----------------------------
# READ AND CLEAN CSV DATA
# -----------------------------
people_df = pd.read_csv('dataset/people.csv')

# Clean column names, just in case
people_df.columns = (
    people_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# Convert datetime columns
#orders_df["ship_date"] = pd.to_datetime(orders_df["ship_date"])
#orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])
people_df.head()

,region,person
0,Central,Anna Andreadi
1,South,Chuck Magee
2,East,Kelly Williams
3,West,Matt Collister
4,Africa,Deborah Brumfield


In [16]:
# -----------------------------
# CREATE TABLE
# -----------------------------

# Create table
create_table_sql = """
CREATE TABLE people (
    region VARCHAR(100),
    person VARCHAR(100)
);
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()

# Insert CSV data
people_df.to_sql(
    "people",
    engine,
    if_exists="append",
    index=False
)

print("CSV data loaded into people table successfully.")

CSV data loaded into people table successfully.


In [17]:
query = "SELECT * FROM people LIMIT 5;"

df_check = pd.read_sql(query, engine)

print(df_check)

    region             person
0  Central      Anna Andreadi
1    South        Chuck Magee
2     East     Kelly Williams
3     West     Matt Collister
4   Africa  Deborah Brumfield


In [18]:
# -----------------------------
# READ AND CLEAN CSV DATA
# -----------------------------
products_df = pd.read_csv('dataset/products.csv')

# Clean column names, just in case
products_df.columns = (
    products_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# Convert datetime columns
#orders_df["ship_date"] = pd.to_datetime(orders_df["ship_date"])
#orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])
products_df.head()

,product_id,category,sub_category,product_name
0,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...
1,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black"
2,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID"
3,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless"
4,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed"


In [19]:
# -----------------------------
# CREATE TABLE
# -----------------------------

# Create table
create_table_sql = """
CREATE TABLE products (
    product_id VARCHAR(30) PRIMARY KEY,
    category VARCHAR(50) NOT NULL,
    sub_category VARCHAR(50) NOT NULL,
    product_name VARCHAR(255) NOT NULL
);
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()

# Insert CSV data
products_df.to_sql(
    "products",
    engine,
    if_exists="append",
    index=False
)

print("CSV data loaded into products table successfully.")

CSV data loaded into products table successfully.


In [20]:
query = "SELECT * FROM products LIMIT 5;"

df_check = pd.read_sql(query, engine)

print(df_check)

         product_id    category sub_category  \
0   TEC-AC-10003033  Technology  Accessories   
1   FUR-CH-10003950   Furniture       Chairs   
2   TEC-PH-10004664  Technology       Phones   
3   TEC-PH-10004583  Technology       Phones   
4  TEC-SHA-10000501  Technology      Copiers   

                                        product_name  
0  Plantronics CS510 - Over-the-Head monaural Wir...  
1          Novimex Executive Leather Armchair, Black  
2                  Nokia Smart Phone, with Caller ID  
3                     Motorola Smart Phone, Cordless  
4                     Sharp Wireless Fax, High-Speed  


In [21]:
# -----------------------------
# READ AND CLEAN CSV DATA
# -----------------------------
returned_orders_df = pd.read_csv('dataset/returned_orders.csv')

# Clean column names, just in case
returned_orders_df.columns = (
    returned_orders_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# Convert datetime columns
#orders_df["ship_date"] = pd.to_datetime(orders_df["ship_date"])
#orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])
returned_orders_df.head()

,returned,order_id,market
0,Yes,MX-2013-168137,LATAM
1,Yes,US-2011-165316,LATAM
2,Yes,ES-2013-1525878,EU
3,Yes,CA-2013-118311,United States
4,Yes,ES-2011-1276768,EU


In [22]:
# -----------------------------
# CREATE TABLE
# -----------------------------

# Create table
create_table_sql = """
CREATE TABLE returned_orders (
    returned VARCHAR(10),
    order_id VARCHAR(50),
    market VARCHAR(100)
);
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()

# Insert CSV data
returned_orders_df.to_sql(
    "returned_orders",
    engine,
    if_exists="append",
    index=False
)

print("CSV data loaded into returned_orders table successfully.")

CSV data loaded into returned_orders table successfully.


In [23]:
query = "SELECT * FROM returned_orders LIMIT 5;"

df_check = pd.read_sql(query, engine)

print(df_check)

  returned         order_id         market
0      Yes   MX-2013-168137          LATAM
1      Yes   US-2011-165316          LATAM
2      Yes  ES-2013-1525878             EU
3      Yes   CA-2013-118311  United States
4      Yes  ES-2011-1276768             EU
